# InfiniteTalk on Kaggle — validation rig

**Purpose:** answer ONE question as cheaply as possible: *does the budget config
(quantized weights + low-VRAM mode + 4-step LoRA) produce acceptable quality on our composite?*
Not production. Production config transplants to RunPod unchanged — only the machine changes.

**Before running:** Settings (right panel) → Accelerator = **GPU T4 x2** · Internet = **On** ·
Add-ons → Secrets → add `HF_TOKEN` (your Hugging Face read token).

**Expect:** weights download ~30–45 min (first run only — see the dataset-caching cell),
render of a 10s clip anywhere from 5–25 min on a T4. Dependency errors on first run are NORMAL;
fix, restart kernel, re-run.

In [ ]:
# ============ CELL 1: CONFIG — the only cell you should need to edit ============
# Every choice lives here so the exact same config moves to RunPod later.

RESOLUTION   = "infinitetalk-480"   # 480p first; "infinitetalk-720" once 480p works
SAMPLE_STEPS = 4                     # 4 with lightx2v LoRA; 40 for full quality (SLOW on T4)
USE_LORA     = True                  # lightx2v 4-step distill LoRA
QUANT        = "int8"                # "int8" for T4 (older GPU). "fp8" needs newer cards (RunPod 4090).
LOW_VRAM     = True                  # --num_persistent_param_in_dit 0
# Guidance scales: repo README says text 5 / audio 4 WITHOUT LoRA, text 1 / audio 2 WITH LoRA
TEXT_GUIDE   = 1 if USE_LORA else 5
AUDIO_GUIDE  = 2 if USE_LORA else 4

PROMPT = "a person giving a friendly talk to camera, natural gestures"
IMAGE_PATH = "/kaggle/input/datasets/utkarsh754/p2t-test-photo/photo.jpg"  # your attached dataset (check exact path with: !ls -R /kaggle/input/)
AUDIO_PATH = "/kaggle/working/input/audio.wav"   # generated by cell 1.5

import os
os.makedirs("/kaggle/working/input", exist_ok=True)
W = "/tmp/weights"   # weights live on the big scratch disk, NOT the 19.5GB working dir
print("Config OK.")

In [ ]:
# ============ CELL 1.5: no audio file? generate one (same TTS as prompt2tube v1) ============
# Skip this cell if you uploaded a real audio.wav.
SCRIPT = ("Hi, I'm testing our new video pipeline. This voice was generated for free, "
          "and this face is animated by an open source model running on a free GPU. "
          "If you can see natural movement while I speak, the experiment worked.")

!pip install -q edge-tts
with open("/kaggle/working/input/script.txt", "w") as f:
    f.write(SCRIPT)
# CLI avoids Jupyter's already-running event loop; --file avoids shell-quoting issues
!edge-tts --voice en-US-ChristopherNeural --file /kaggle/working/input/script.txt --write-media /kaggle/working/input/audio.mp3
# Convert mp3 -> 16 kHz mono wav (the shape speech models expect)
!ffmpeg -y -loglevel error -i /kaggle/working/input/audio.mp3 -ar 16000 -ac 1 /kaggle/working/input/audio.wav
import os; print("audio.wav ready:", os.path.getsize('/kaggle/working/input/audio.wav'), "bytes (~15s of speech)")

In [ ]:
# ============ CELL 2: clone repo + dependencies ============
# Kaggle ships torch+CUDA already — do NOT reinstall torch, it wastes 20 min and breaks CUDA.
%cd /kaggle/working
!git clone https://github.com/MeiGen-AI/InfiniteTalk 2>/dev/null || echo 'already cloned'
%cd InfiniteTalk

# Install their requirements MINUS heavyweight/preinstalled ones; add what Kaggle lacks.
!grep -viE 'torch|flash' requirements.txt > req_kaggle.txt || true
!pip install -q -r req_kaggle.txt
!pip install -q librosa soundfile einops omegaconf ftfy
!pip install -q -U huggingface_hub  # newest CLI ('hf'), also silences its interactive update prompt
# NOTE: we skip flash-attn (hour-long compile on T4, often fails on sm_75).
# If generation errors mentioning flash_attn, look for an --attn or sdpa fallback flag,
# or: !pip install -q flash-attn --no-build-isolation   (last resort, expect ~40 min)
print("Deps done")

In [ ]:
# ============ CELL 3: download weights (~40 GB, SELECTIVE — the full repos are 82 GB and 169 GB!) ============
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
W = "/tmp/weights"   # /tmp = big scratch disk; /kaggle/working only holds 19.5 GB

# Base repo WITHOUT the 65 GB of DiT shards — the int8 quant file below replaces the DiT.
!hf download Wan-AI/Wan2.1-I2V-14B-480P \
  --include "models_t5_umt5-xxl-enc-bf16.pth" --include "Wan2.1_VAE.pth" \
  --include "models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth" \
  --include "config.json" --include "diffusion_pytorch_model.safetensors.index.json" \
  --include "google/*" --include "xlm-roberta-large/*" \
  --local-dir {W}/Wan2.1-I2V-14B-480P

# Audio encoder (small)
!hf download TencentGameMate/chinese-wav2vec2-base --local-dir {W}/chinese-wav2vec2-base
!hf download TencentGameMate/chinese-wav2vec2-base model.safetensors --revision refs/pr/1 --local-dir {W}/chinese-wav2vec2-base

# InfiniteTalk: ONLY single-person weights + int8 quant pair (.safetensors + .json)
!hf download MeiGen-AI/InfiniteTalk --include "single/*" --include "quant_models/infinitetalk_single_int8.*" \
  --local-dir {W}/InfiniteTalk

# 4-step distill LoRA
!hf download Kijai/WanVideo_comfy Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank32.safetensors --local-dir {W}/lora

!du -sh {W}/*
!df -h /tmp | tail -1   # sanity: make sure the disk still has headroom

**⏱ Session reality:** weights live in `/tmp`, which is wiped on every session restart —
so each fresh session re-pays this download (~30–45 min). That is the Kaggle tax on a
validation rig; the persistent-disk fix is what RunPod is for. After any restart, the
run order is always: **1 → 1.5 → 2 → 3 → 4 → 5 → 6**. Only attached Input datasets survive.

In [ ]:
# ============ CELL 4: input JSON (the script's way of taking photo+audio) ============
import json
spec = {
    "prompt": PROMPT,
    "cond_video": IMAGE_PATH,          # yes, 'video' — a single image is accepted for I2V
    "cond_audio": {"person1": AUDIO_PATH},
}
with open("/kaggle/working/input.json", "w") as f:
    json.dump(spec, f, indent=2)
print(json.dumps(spec, indent=2))
assert os.path.exists(IMAGE_PATH), "Upload photo.jpg first (see cell 1)"
assert os.path.exists(AUDIO_PATH), "Upload audio.wav first (see cell 1)"

In [ ]:
# ============ CELL 5: render (budget config assembled from cell 1) ============
W = "/tmp/weights"
cmd = [
    "python", "generate_infinitetalk.py",
    "--ckpt_dir", f"{W}/Wan2.1-I2V-14B-480P",
    "--wav2vec_dir", f"{W}/chinese-wav2vec2-base",
    "--infinitetalk_dir", f"{W}/InfiniteTalk/single/infinitetalk.safetensors",
    "--input_json", "/kaggle/working/input.json",
    "--size", RESOLUTION,
    "--sample_steps", str(SAMPLE_STEPS),
    "--sample_text_guide_scale", str(TEXT_GUIDE),
    "--sample_audio_guide_scale", str(AUDIO_GUIDE),
    "--motion_frame", "9",
    "--mode", "streaming",
    "--save_file", "/kaggle/working/result",
]
if LOW_VRAM:
    cmd += ["--num_persistent_param_in_dit", "0"]
if QUANT:
    cmd += ["--quant", QUANT,
            "--quant_dir", f"{W}/InfiniteTalk/quant_models/infinitetalk_single_{QUANT}.safetensors"]
if USE_LORA:
    cmd += ["--lora_dir", f"{W}/lora/Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank32.safetensors",
            "--lora_scale", "1.0"]
print(" ".join(cmd))
!{' '.join(cmd)}

In [ ]:
# ============ CELL 6: watch the result + record the verdict ============
from IPython.display import Video, display
import glob, time
outs = sorted(glob.glob("/kaggle/working/result*.mp4"), key=os.path.getmtime)
assert outs, "No output video found — check cell 5's log for the real error"
display(Video(outs[-1], embed=True, width=480))

# The four things to write down for the vault / senior demo:
print(f"""VERDICT SHEET
  config   : {QUANT} quant, {SAMPLE_STEPS} steps, LoRA={USE_LORA}, {RESOLUTION}, low_vram={LOW_VRAM}
  output   : {outs[-1]}
  Judge vs the $1.45 HeyGen Avatar IV render and free MoDA render:
  1. lip sync accuracy?  2. body/gesture naturalness?  3. identity preserved?  4. artifacts/color?
""")

## Known failure modes (so you don't panic)

| Symptom | Meaning | Fix |
|---|---|---|
| `CUDA out of memory` | T4's 16 GB exceeded | confirm `QUANT='int8'`, `LOW_VRAM=True`, 480p; shorten audio to ≤10 s |
| Killed / process dies silently | system RAM exceeded during load | restart, run cell 5 alone; keep quant on |
| `fp8 not supported` / dtype error | T4 (sm_75) can't do fp8 compute | that's why default is int8; fp8 is for RunPod 4090 later |
| `flash_attn` import error | we skipped it deliberately | look for sdpa/attn fallback flag; last resort install (40 min) |
| Download quota / 401 on HF | token missing or gated repo | check HF_TOKEN secret; accept model terms on the HF page |
| Render absurdly slow (>60 min for 10 s) | fell back to full steps or CPU | confirm SAMPLE_STEPS=4 printed in cell 5's command |

**Session discipline:** Kaggle gives ~30 GPU-h/week. Weights download burns GPU session time —
do the dataset-caching step after your first successful run, and develop with GPU **off**
(Settings → Accelerator → None) whenever you're editing cells, not rendering.